## [dq01]  kolomwaarde is niet leeg

In [ ]:
def dq01_kolomwaarde_is_niet_leeg(laag_1, schema_1, tabel_1, sleutelkolom_1, kolom_1):
    dq_code = "01"
    dq_beschrijving = "kolomwaarde is leeg"
    # voeg een kolom toe met postfix _dq01
    con.sql(f"""
        ALTER TABLE {laag_1}.{schema_1}.{tabel_1} ADD IF NOT EXISTS {kolom_1}_dq{dq_code} boolean;
    """)

    # shrijf het datakwaliteit meetresultaat weg in de {kolom_1}_dq01 kolom: False => waarde is leeg of NULL; True =>  waarde is NIET leeg of NULL
    con.sql(f"""
        UPDATE {laag_1}.{schema_1}.{tabel_1}
        SET {kolom_1}_dq{dq_code} = CASE WHEN {kolom_1} IS NULL THEN False
                                     WHEN {kolom_1} = '' THEN False
                                     ELSE True END;
    """)

    # shrijf het datakwaliteit meetresultaat weg naar csv bestand
    con.sql(f"""
        COPY (
            SELECT NOW() AS dq_timestamp, 'dq{dq_code}' as dq_code, '{dq_beschrijving}' as dq_beschrijving, {sleutelkolom_1}, {kolom_1} FROM {laag_1}.{schema_1}.{tabel_1}
            WHERE {kolom_1}_dq{dq_code} = False
        ) 
        TO './datakwaliteit_meetwaarden/{laag_1}.{schema_1}.{tabel_1}.{kolom_1}_dq{dq_code}.csv'  CSV DELIMITER ';' HEADER;
    """)

    # schrijven het resultaat van de datakwaliteit meting weg naar het datakwaliteit log tabel
    con.sql(f"""
        INSERT INTO {laag_1}.datakwaliteit.log ( timestamp, dq_code, laag_1, schema_1, tabel_1, sleutelkolom_1, kolom_1, meetwaarde_true_1, meetwaarde_false_1, meetwaarde_totaal_1 )
            VALUES ( NOW(), '{dq_code}', '{laag_1}', '{schema_1}', '{tabel_1}', '{sleutelkolom_1}', '{kolom_1}', (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = True), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = False), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1}) );
    """)



## [dq02]  kolomwaarde is tussen kleiner_dan en groter_dan

In [ ]:
def dq02_kolomwaarde_is_tussen(laag_1, schema_1, tabel_1, sleutelkolom_1, kolom_1, kleiner_dan, groter_dan):
    dq_code = "02"
    dq_beschrijving = f"kolomwaarde is niet kleiner dan {kleiner_dan} en groter dan {groter_dan}"
    meetvariabele_1a = str(kleiner_dan)
    meetvariabele_1b = str(groter_dan)
    
    # voeg een kolom toe met postfix _dq02
    con.sql(f"""
        ALTER TABLE {laag_1}.{schema_1}.{tabel_1} ADD IF NOT EXISTS {kolom_1}_dq{dq_code} boolean;
    """)

    # shrijf het datakwaliteit meetresultaat weg in de {kolom_1}_dq02 kolom: True => waarde is groter dan en kleiner dan;  => False waarde is NIET groter dan en kleiner dan
    con.sql(f"""
        UPDATE {laag_1}.{schema_1}.{tabel_1}
        SET {kolom_1}_dq{dq_code} = CASE WHEN {kolom_1} > {groter_dan} AND {kolom_1} < {kleiner_dan} THEN True
                                     ELSE False END;
    """)

    # shrijf het datakwaliteit meetresultaat weg naar csv bestand
    con.sql(f"""
        COPY (
            SELECT NOW() AS dq_timestamp, 'dq{dq_code}' as dq_code, '{dq_beschrijving}' as dq_beschrijving, {sleutelkolom_1}, {kolom_1} FROM {laag_1}.{schema_1}.{tabel_1}
            WHERE {kolom_1}_dq{dq_code} = False
        ) 
        TO './datakwaliteit_meetwaarden/{laag_1}.{schema_1}.{tabel_1}.{kolom_1}_dq{dq_code}.csv'  CSV DELIMITER ';' HEADER;
    """)

    # schrijven het resultaat van de datakwaliteit meting weg naar het datakwaliteit log tabel
    con.sql(f"""
        INSERT INTO {laag_1}.datakwaliteit.log ( timestamp, dq_code, laag_1, schema_1, tabel_1, sleutelkolom_1, kolom_1,meetvariabele_1a, meetvariabele_1b, meetwaarde_true_1, meetwaarde_false_1, meetwaarde_totaal_1 )
            VALUES ( NOW(), '{dq_code}', '{laag_1}', '{schema_1}', '{tabel_1}', '{sleutelkolom_1}', '{kolom_1}', '{meetvariabele_1a}', '{meetvariabele_1b}', (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = True), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = False), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1}) );
    """)

## [dq03]  kolomwaarde is conform patroon

In [ ]:
def dq03_kolomwaarde_heeft_patroon(laag_1, schema_1, tabel_1, sleutelkolom_1, kolom_1, patroon):
    meetvariabele_1a = str(patroon)
    dq_code = "03"
    dq_beschrijving = "kolomwaarde is niet conform patroon"

    # voeg een kolom toe met postfix _dq03
    con.sql(f"""
        ALTER TABLE {laag_1}.{schema_1}.{tabel_1} ADD IF NOT EXISTS {kolom_1}_dq{dq_code} boolean
    """)
    
    # shrijf het datakwaliteit meetresultaat weg in de {kolom_1}_dq02 kolom: True => waarde is conform het patroon;  => False waarde is NIET conform het patroon
    con.sql(f"""
        UPDATE {laag_1}.{schema_1}.{tabel_1}
        SET {kolom_1}_dq{dq_code} = CASE WHEN regexp_matches({kolom_1}, '{patroon}') THEN True
                                     ELSE False END;
    """)

    # shrijf het datakwaliteit meetresultaat weg naar csv bestand
    con.sql(f"""
        COPY (
            SELECT NOW() AS dq_timestamp, 'dq{dq_code}' as dq_code, '{dq_beschrijving}' as dq_beschrijving, {sleutelkolom_1}, {kolom_1} FROM {laag_1}.{schema_1}.{tabel_1}
            WHERE {kolom_1}_dq{dq_code} = False
        ) 
        TO './datakwaliteit_meetwaarden/{laag_1}.{schema_1}.{tabel_1}.{kolom_1}_dq{dq_code}.csv'  CSV DELIMITER ';' HEADER;
    """)

    # schrijven het rusultaat van de datakwaliteit meting weg naar het datakwaliteit log tabel
    con.sql(f"""
        INSERT INTO {laag_1}.datakwaliteit.log ( timestamp, dq_code, laag_1, schema_1, tabel_1, sleutelkolom_1, kolom_1,meetvariabele_1a, meetwaarde_true_1, meetwaarde_false_1, meetwaarde_totaal_1 )
            VALUES ( NOW(), '{dq_code}', '{laag_1}', '{schema_1}', '{tabel_1}', '{sleutelkolom_1}', '{kolom_1}', '{meetvariabele_1a}', (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = True), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = False), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1}) );
    """)

## [dq04]  kolomwaarde begint niet met

In [ ]:
def dq04_kolomwaarde_begint_niet_met(laag_1, schema_1, tabel_1, sleutelkolom_1, kolom_1, patroon):
    meetvariabele_1a = str(patroon)
    dq_code = "04"
    dq_beschrijving = f"kolomwaarde begint met {patroon}".replace("|", " of ").replace("^[","[")

    # voeg een kolom toe met postfix _dq04
    con.sql(f"""
        ALTER TABLE {laag_1}.{schema_1}.{tabel_1} ADD IF NOT EXISTS {kolom_1}_dq{dq_code} boolean
    """)
    
    # shrijf het datakwaliteit meetresultaat weg in de {kolom_1}_dq02 kolom: True => waarde is conform het patroon;  => False waarde is NIET conform het patroon
    con.sql(f"""
        UPDATE {laag_1}.{schema_1}.{tabel_1}
        SET {kolom_1}_dq{dq_code} = CASE WHEN regexp_matches({kolom_1}, '{patroon}') THEN False
                                     ELSE True END;
    """)

    # shrijf het datakwaliteit meetresultaat weg naar csv bestand
    con.sql(f"""
        COPY (
            SELECT NOW() AS dq_timestamp, 'dq{dq_code}' as dq_code, '{dq_beschrijving}' as dq_beschrijving, {sleutelkolom_1}, {kolom_1} FROM {laag_1}.{schema_1}.{tabel_1}
            WHERE {kolom_1}_dq{dq_code} = False
        ) 
        TO './datakwaliteit_meetwaarden/{laag_1}.{schema_1}.{tabel_1}.{kolom_1}_dq{dq_code}.csv'  CSV DELIMITER ';' HEADER;
    """)

    # schrijven het resultaat van de datakwaliteit meting weg naar het datakwaliteit log tabel
    con.sql(f"""
        INSERT INTO {laag_1}.datakwaliteit.log ( timestamp, dq_code, laag_1, schema_1, tabel_1, sleutelkolom_1, kolom_1,meetvariabele_1a, meetwaarde_true_1, meetwaarde_false_1, meetwaarde_totaal_1 )
            VALUES ( NOW(), '{dq_code}', '{laag_1}', '{schema_1}', '{tabel_1}', '{sleutelkolom_1}', '{kolom_1}', '{meetvariabele_1a}', (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = True), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = False), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1}) );
    """)

## [dq05]  kolomwaarde eindigt niet op

In [ ]:
def dq05_kolomwaarde_eindigt_niet_op(laag_1, schema_1, tabel_1, sleutelkolom_1, kolom_1, patroon):
    meetvariabele_1a = str(patroon)
    dq_code = "05"
    dq_beschrijving = f"kolomwaarde eindigd op {patroon}".replace("|", " of ").replace("]$","]")

    # voeg een kolom toe met postfix _dq05
    con.sql(f"""
        ALTER TABLE {laag_1}.{schema_1}.{tabel_1} ADD IF NOT EXISTS {kolom_1}_dq{dq_code} boolean
    """)
    
    # shrijf het datakwaliteit meetresultaat weg in de {kolom_1}_dq02 kolom: True => waarde is conform het patroon;  => False waarde is NIET conform het patroon
    con.sql(f"""
        UPDATE {laag_1}.{schema_1}.{tabel_1}
        SET {kolom_1}_dq{dq_code} = CASE WHEN regexp_matches({kolom_1}, '{patroon}') THEN False
                                     ELSE True END;
    """)

    # shrijf het datakwaliteit meetresultaat weg naar csv bestand
    con.sql(f"""
        COPY (
            SELECT NOW() AS dq_timestamp, 'dq{dq_code}' as dq_code, '{dq_beschrijving}' as dq_beschrijving, {sleutelkolom_1}, {kolom_1} FROM {laag_1}.{schema_1}.{tabel_1}
            WHERE {kolom_1}_dq{dq_code} = False
        ) 
        TO './datakwaliteit_meetwaarden/{laag_1}.{schema_1}.{tabel_1}.{kolom_1}_dq{dq_code}.csv'  CSV DELIMITER ';' HEADER;
    """)

    # schrijven het resultaat van de datakwaliteit meting weg naar het datakwaliteit log tabel
    con.sql(f"""
        INSERT INTO {laag_1}.datakwaliteit.log ( timestamp, dq_code, laag_1, schema_1, tabel_1, sleutelkolom_1, kolom_1,meetvariabele_1a, meetwaarde_true_1, meetwaarde_false_1, meetwaarde_totaal_1 )
            VALUES ( NOW(), '{dq_code}', '{laag_1}', '{schema_1}', '{tabel_1}', '{sleutelkolom_1}', '{kolom_1}', '{meetvariabele_1a}', (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = True), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = False), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1}) );
    """)

## [dq06]  kolomwaarde is korter dan en langer dan

In [ ]:
def dq06_kolomwaarde_heeft_lengte(laag_1, schema_1, tabel_1, sleutelkolom_1, kolom_1, korter_dan, langer_dan):
    dq_code = "06"
    dq_beschrijving = f"kolomwaarde is niet langer dan {langer_dan} en korter dan {korter_dan}"
    meetvariabele_1a = str(korter_dan)
    meetvariabele_1b = str(langer_dan)
    
    # voeg een kolom toe met postfix _dq06
    con.sql(f"""
        ALTER TABLE {laag_1}.{schema_1}.{tabel_1} ADD IF NOT EXISTS {kolom_1}_dq{dq_code} boolean;
    """)

    # shrijf het datakwaliteit meetresultaat weg in de {kolom_1}_dq02 kolom: True => waarde is groter dan en kleiner dan;  => False waarde is NIET groter dan en kleiner dan
    con.sql(f"""
        UPDATE {laag_1}.{schema_1}.{tabel_1}
        SET {kolom_1}_dq{dq_code} = CASE WHEN len({kolom_1}) < {korter_dan} AND len({kolom_1}) > {langer_dan} THEN True
                                     ELSE False END;
    """)

    # shrijf het datakwaliteit meetresultaat weg naar csv bestand
    con.sql(f"""
        COPY (
            SELECT NOW() AS dq_timestamp, 'dq{dq_code}' as dq_code, '{dq_beschrijving}' as dq_beschrijving, {sleutelkolom_1}, {kolom_1} FROM {laag_1}.{schema_1}.{tabel_1}
            WHERE {kolom_1}_dq{dq_code} = False
        ) 
        TO './datakwaliteit_meetwaarden/{laag_1}.{schema_1}.{tabel_1}.{kolom_1}_dq{dq_code}.csv'  CSV DELIMITER ';' HEADER;
    """)

    # schrijven het resultaat van de datakwaliteit meting weg naar het datakwaliteit log tabel
    con.sql(f"""
        INSERT INTO {laag_1}.datakwaliteit.log ( timestamp, dq_code, laag_1, schema_1, tabel_1, sleutelkolom_1, kolom_1,meetvariabele_1a, meetvariabele_1b, meetwaarde_true_1, meetwaarde_false_1, meetwaarde_totaal_1 )
            VALUES ( NOW(), '{dq_code}', '{laag_1}', '{schema_1}', '{tabel_1}', '{sleutelkolom_1}', '{kolom_1}', '{meetvariabele_1a}', '{meetvariabele_1b}', (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = True), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = False), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1}) );
    """)

## [dq07]  kolomwaarde is niet in lijst

In [ ]:
def dq07_kolomwaarde_niet_in_lijst(laag_1, schema_1, tabel_1, sleutelkolom_1, kolom_1, laag_2, schema_2, tabel_2, kolom_2):
    dq_code = "07"
    dq_beschrijving = f"kolomwaarde is niet in lijst"
    
    # voeg een kolom toe met postfix _dq07
    con.sql(f"""
        ALTER TABLE {laag_1}.{schema_1}.{tabel_1} ADD IF NOT EXISTS {kolom_1}_dq{dq_code} boolean;
    """)

    # shrijf het datakwaliteit meetresultaat weg in de {kolom_1}_dq02 kolom: True => waarde is groter dan en kleiner dan;  => False waarde is NIET groter dan en kleiner dan
    con.sql(f"""
        UPDATE {laag_1}.{schema_1}.{tabel_1}
        SET {kolom_1}_dq{dq_code} = CASE WHEN {kolom_1} IN (SELECT {kolom_2} FROM {laag_2}.{schema_2}.{tabel_2}) THEN True
                                     ELSE False END;
    """)
   
    # shrijf het datakwaliteit meetresultaat weg naar csv bestand
    con.sql(f"""
        COPY (
            SELECT NOW() AS dq_timestamp, 'dq{dq_code}' as dq_code, '{dq_beschrijving}' as dq_beschrijving, {sleutelkolom_1}, {kolom_1} FROM {laag_1}.{schema_1}.{tabel_1}
            WHERE {kolom_1}_dq{dq_code} = False
        ) 
        TO './datakwaliteit_meetwaarden/{laag_1}.{schema_1}.{tabel_1}.{kolom_1}_dq{dq_code}.csv'  CSV DELIMITER ';' HEADER;
    """)

    # schrijven het resultaat van de datakwaliteit meting weg naar het datakwaliteit log tabel
    con.sql(f"""
        INSERT INTO {laag_1}.datakwaliteit.log ( timestamp, dq_code, laag_1, schema_1, tabel_1, sleutelkolom_1, kolom_1, meetwaarde_true_1, meetwaarde_false_1, meetwaarde_totaal_1, laag_2, schema_2, tabel_2, kolom_2)
            VALUES ( NOW(), '{dq_code}', '{laag_1}', '{schema_1}', '{tabel_1}', '{sleutelkolom_1}', '{kolom_1}', (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = True), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = False), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1}), '{laag_2}', '{schema_2}', '{tabel_2}', '{kolom_2}' );
    """)

## [dq08]  kolom is niet leeg

In [ ]:
def dq08_kolom_is_niet_leeg(laag_1, schema_1, tabel_1, kolom_1):
    dq_code = "08"
    dq_beschrijving = "alle kolomwaarden zijn leeg"
    
    # voeg een kolom toe met postfix _dq08
    con.sql(f"""
        ALTER TABLE {laag_1}.{schema_1}.{tabel_1} ADD IF NOT EXISTS {kolom_1}_dq{dq_code} boolean;
    """)

    # shrijf het datakwaliteit meetresultaat weg in de {kolom_1}_dq01 kolom: False => waarde is leeg of NULL; True =>  waarde is NIET leeg of NULL
    con.sql(f"""
        UPDATE {laag_1}.{schema_1}.{tabel_1}
        SET {kolom_1}_dq{dq_code} = CASE WHEN {kolom_1} IS NULL THEN False
                                     WHEN {kolom_1} = '' THEN False
                                     ELSE True END;
    """)

    # het datakwaliteit meetresultaat wordt niet weggeschreven naar een csv bestand

    # schrijven het resultaat van de datakwaliteit meting weg naar het datakwaliteit log tabel
    con.sql(f"""
        INSERT INTO {laag_1}.datakwaliteit.log ( timestamp, dq_code, laag_1, schema_1, tabel_1, kolom_1, meetwaarde_true_1, meetwaarde_false_1, meetwaarde_totaal_1 )
            VALUES ( NOW(), '{dq_code}', '{laag_1}', '{schema_1}', '{tabel_1}', '{kolom_1}', (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = True), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1} WHERE {kolom_1}_dq{dq_code} = False), (SELECT COUNT(*) FROM {laag_1}.{schema_1}.{tabel_1}) );
    """)



In [ ]:
print("Er zijn 8 User Defined functions geladen")